In [6280]:
import pandas as pd
import regex as re
import io

In [6281]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

In [6282]:
df = read_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual
df['AF'] = df['INFO'].apply(extract_af)

df.head(60)

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND...",0.987559
6,Human,121,.,C,G,49314,PASS,"DP=3853;AF=0.998962;SB=0;DP4=0,0,3779,70",0.998962
7,Human,209,.,C,G,2469,PASS,"DP=4542;AF=0.109203;SB=2;DP4=3584,459,436,60",0.109203
8,Human,237,.,G,GC,49314,PASS,"DP=4988;AF=0.966921;SB=14;DP4=164,21,3988,835;...",0.966921
9,Human,270,.,C,CG,49314,PASS,"DP=5712;AF=0.945553;SB=6;DP4=242,88,4122,1279;...",0.945553


In [6283]:
type(df['AF'].iloc[0])

str

In [6284]:
#get sequnce of individual at each position
og_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/og_ref.csv')

og_ref.shape

(13314, 3)

In [6285]:
type(og_ref['POS'].iloc[30])

numpy.int64

In [6286]:
new_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/1000_genome_new_ref/1000_genome_new_ref_v2.csv')

new_ref.shape

(13314, 3)

In [6287]:
df = df[['POS', 'REF', 'ALT', 'AF']]  

print(og_ref.columns)
print(df.columns)

merged_df = pd.merge(og_ref, df, on='POS', how='left', suffixes=('_og', '_vcf'))

merged_df.rename(columns={'REF_og':'ALT_og', 'ALT':'ALT_vcf'}, inplace=True)

merged_df.head(33)

Index(['POS', 'Type', 'REF'], dtype='object')
Index(['POS', 'REF', 'ALT', 'AF'], dtype='object')


,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
0,1,sequence,G,NaN,NaN,NaN
1,2,sequence,C,NaN,NaN,NaN
2,3,sequence,T,NaN,NaN,NaN
3,4,sequence,G,NaN,NaN,NaN
4,5,sequence,A,NaN,NaN,NaN
5,6,sequence,C,NaN,NaN,NaN
6,7,sequence,A,NaN,NaN,NaN
7,8,sequence,C,NaN,NaN,NaN
8,9,sequence,G,NaN,NaN,NaN
9,10,sequence,C,NaN,NaN,NaN


In [6288]:
merged_df.shape

(13346, 6)

In [6289]:
merged_df['POS'].iloc[30]

31

In [6290]:
merged_df[(merged_df['POS'] - 1)!= merged_df.index]

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
386,386,sequence,T,T,TGGCC,0.736424
387,387,sequence,G,NaN,NaN,NaN
388,388,sequence,G,NaN,NaN,NaN
389,389,sequence,C,NaN,NaN,NaN
390,390,sequence,C,NaN,NaN,NaN
...,...,...,...,...,...,...
13341,13310,sequence,T,NaN,NaN,NaN
13342,13311,sequence,C,NaN,NaN,NaN
13343,13312,sequence,G,NaN,NaN,NaN
13344,13313,sequence,C,NaN,NaN,NaN


In [6291]:
merged_df.head(389)

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
0,1,sequence,G,NaN,NaN,NaN
1,2,sequence,C,NaN,NaN,NaN
2,3,sequence,T,NaN,NaN,NaN
3,4,sequence,G,NaN,NaN,NaN
4,5,sequence,A,NaN,NaN,NaN
...,...,...,...,...,...,...
384,385,sequence,C,NaN,NaN,NaN
385,386,sequence,T,T,TGGCAA,0.100669
386,386,sequence,T,T,TGGCC,0.736424
387,387,sequence,G,NaN,NaN,NaN


In [6292]:
df[df['POS'] == 386]

,POS,REF,ALT,AF
13,386,T,TGGCAA,0.100669
14,386,T,TGGCC,0.736424


In [6293]:
# Combine the ALT_og column and ALT_vcf into ALT_combined, and set AF to 0 where variant did not exist
merged_df['ALT_combined'] = merged_df.apply(lambda x: x['ALT_vcf'] if pd.notna(x['ALT_vcf']) else x['ALT_og'], axis=1)
merged_df['AF'] = merged_df['AF'].fillna(0) 

merged_df.head(32)

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
5,6,sequence,C,NaN,NaN,0,C
6,7,sequence,A,NaN,NaN,0,A
7,8,sequence,C,NaN,NaN,0,C
8,9,sequence,G,NaN,NaN,0,G
9,10,sequence,C,NaN,NaN,0,C


In [6294]:
merged_df.head(80) 

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
...,...,...,...,...,...,...,...
75,76,sequence,G,NaN,NaN,0,G
76,77,sequence,G,NaN,NaN,0,G
77,78,sequence,T,NaN,NaN,0,T
78,79,sequence,G,NaN,NaN,0,G


In [6295]:
merged_df.head(637) 



,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
...,...,...,...,...,...,...,...
632,632,sequence,G,NaN,NaN,0,G
633,633,sequence,G,GT,G,0.052245,G
634,634,sequence,T,NaN,NaN,0,T
635,635,sequence,G,NaN,NaN,0,G


In [6296]:
###to identify deletions from this table 
###-> Alt_og == Alt_vcf (get position)
###-> save position if any of subsequent positions are NaN in Alt_vcf but not Nan in Alt_og 
   ### --> add the base and position to a deletion_dictionary = {104: 'C'}

In [6297]:
merged_df.drop(columns=['ALT_vcf','Type'], inplace=True)

merged_df.rename(columns={'ALT_og': 'og_ref', 'AF':'AF_old', 'ALT_combined':'ALT_old'}, inplace=True)

merged_df.head(73)

,POS,og_ref,REF_vcf,AF_old,ALT_old
0,1,G,NaN,0,G
1,2,C,NaN,0,C
2,3,T,NaN,0,T
3,4,G,NaN,0,G
4,5,A,NaN,0,A
...,...,...,...,...,...
68,69,G,NaN,0,G
69,70,C,NaN,0,C
70,71,C,NaN,0,C
71,72,T,NaN,0,T


In [6298]:
type(merged_df['AF_old'].iloc[30])

str

In [6299]:
merged_df = pd.merge(merged_df, new_ref, on='POS', how='left')
merged_df.drop(columns=['Unnamed: 0'], inplace=True)
merged_df.rename(columns={'1000_genome_new_ref': 'new_ref'}, inplace=True)

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T
13342,13311,C,NaN,0,C,C
13343,13312,G,NaN,0,G,G
13344,13313,C,NaN,0,C,C


In [6300]:
merged_df.head(1676)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
1671,1670,C,NaN,0,C,C
1672,1671,G,GC,0.031593,G,G
1673,1671,G,GCTCCC,0.596132,G,G
1674,1672,C,NaN,0,C,0


In [6301]:
merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
99,100,C,NaN,0,C,C
100,101,C,NaN,0,C,C
101,102,T,NaN,0,T,T
102,103,G,GC,0.987559,G,G


In [6302]:
merged_df.head(636)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
631,631,G,NaN,0,G,G
632,632,G,NaN,0,G,G
633,633,G,GT,0.052245,G,G
634,634,T,NaN,0,T,T


In [6303]:
merged_df.head(1676)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
1671,1670,C,NaN,0,C,C
1672,1671,G,GC,0.031593,G,G
1673,1671,G,GCTCCC,0.596132,G,G
1674,1672,C,NaN,0,C,0


In [6304]:
merged_df[(merged_df['new_ref'].str.len()) > (merged_df['og_ref'].str.len())]

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
79,80,A,A,0.977024,AC,AC
236,237,G,G,0.966921,GC,GC
269,270,C,C,0.945553,CG,CG
274,275,C,C,0.937946,CG,CG
283,284,A,A,0.960138,AC,AC
362,363,G,G,0.905516,GT,GT
385,386,T,T,0.100669,TGGCAA,TGGCC
386,386,T,T,0.736424,TGGCC,TGGCC
419,419,C,C,0.925278,CGT,CGT
517,517,G,G,0.940377,GC,GC


In [6305]:
filtered_df = merged_df[(merged_df['og_ref'] != merged_df['new_ref']) & (merged_df['AF_old'] != 0)]

filtered_df.shape #number of ALT rows being changed

(87, 6)

In [6306]:
filtered_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
30,31,T,T,1.000000,C,C
31,32,C,C,1.000000,T,T
72,73,C,C,0.996952,A,A
73,74,A,A,0.999624,C,C
79,80,A,A,0.977024,AC,AC
...,...,...,...,...,...,...
13181,13150,A,A,0.961884,AC,AC
13250,13219,C,C,0.944088,CG,CG
13309,13278,G,G,0.909971,GGC,GGC
13311,13280,G,G,1.000000,C,C


In [6307]:
merged_df['REF_vcf'].iloc[0]

nan

In [6308]:
#incorporating deletions iff it exists
# merged_df['new_ref'] = merged_df.apply(lambda x: x['REF_vcf'] if pd.notna(x['REF_vcf']) and (len(x['REF_vcf']) > 1)  else x['new_ref'], axis=1)

merged_df.head(1676)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
1671,1670,C,NaN,0,C,C
1672,1671,G,GC,0.031593,G,G
1673,1671,G,GCTCCC,0.596132,G,G
1674,1672,C,NaN,0,C,0


In [6309]:
merged_df.head(635)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
630,630,C,NaN,0,C,C
631,631,G,NaN,0,G,G
632,632,G,NaN,0,G,G
633,633,G,GT,0.052245,G,G


In [6310]:
merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
99,100,C,NaN,0,C,C
100,101,C,NaN,0,C,C
101,102,T,NaN,0,T,T
102,103,G,GC,0.987559,G,G


In [6311]:
#using ALT_old unless reference changed and AF_old != 0
merged_df['ALT_new'] = merged_df.apply(lambda x: x['og_ref'] if (x['AF_old'] != 0) and (x['og_ref'] != x['new_ref']) else x['ALT_old'], axis=1)

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T,T
13342,13311,C,NaN,0,C,C,C
13343,13312,G,NaN,0,G,G,G
13344,13313,C,NaN,0,C,C,C


In [6312]:
merged_df.head(32)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
5,6,C,NaN,0,C,C,C
6,7,A,NaN,0,A,A,A
7,8,C,NaN,0,C,C,C
8,9,G,NaN,0,G,G,G
9,10,C,NaN,0,C,C,C


In [6313]:
merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
99,100,C,NaN,0,C,C,C
100,101,C,NaN,0,C,C,C
101,102,T,NaN,0,T,T,T
102,103,G,GC,0.987559,G,G,G


In [6314]:
merged_df.head(636)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
631,631,G,NaN,0,G,G,G
632,632,G,NaN,0,G,G,G
633,633,G,GT,0.052245,G,G,G
634,634,T,NaN,0,T,T,T


In [6315]:
filtered_df2 = merged_df[(merged_df['og_ref'] != merged_df['new_ref'])]

filtered_df2.shape #number of AF rows being changed

(110, 7)

In [6316]:
#using AF_old unless reference changed 
merged_df['AF_new'] = merged_df.apply(lambda x: 1 - float(x['AF_old']) if (x['og_ref'] != x['new_ref']) else float(x['AF_old']), axis=1) #note: 'AF_new' is all in float, 'AF_old' was in string and 0 was integer

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,NaN,0,G,G,G,0.0
1,2,C,NaN,0,C,C,C,0.0
2,3,T,NaN,0,T,T,T,0.0
3,4,G,NaN,0,G,G,G,0.0
4,5,A,NaN,0,A,A,A,0.0
...,...,...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T,T,0.0
13342,13311,C,NaN,0,C,C,C,0.0
13343,13312,G,NaN,0,G,G,G,0.0
13344,13313,C,NaN,0,C,C,C,0.0


In [6317]:
merged_df.iloc[30, :]

POS              31
og_ref            T
REF_vcf           T
AF_old     1.000000
ALT_old           C
new_ref           C
ALT_new           T
AF_new          0.0
Name: 30, dtype: object

In [6318]:
merged_df.head(82)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,NaN,0,G,G,G,0.000000
1,2,C,NaN,0,C,C,C,0.000000
2,3,T,NaN,0,T,T,T,0.000000
3,4,G,NaN,0,G,G,G,0.000000
4,5,A,NaN,0,A,A,A,0.000000
...,...,...,...,...,...,...,...,...
77,78,T,NaN,0,T,T,T,0.000000
78,79,G,NaN,0,G,G,G,0.000000
79,80,A,A,0.977024,AC,AC,A,0.022976
80,81,C,NaN,0,C,C,C,0.000000


In [6319]:
#deal w deletions here

merged_df[merged_df['REF_vcf'].str.len() > 1]

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
102,103,G,GC,0.987559,G,G,G,0.987559
537,537,T,TG,0.957346,T,T,T,0.957346
633,633,G,GT,0.052245,G,G,G,0.052245
763,763,C,CGG,0.649318,C,C,C,0.649318
963,963,T,TGAGAC,0.008979,T,T,T,0.008979
...,...,...,...,...,...,...,...,...
11093,11069,G,GC,0.935852,G,G,G,0.935852
11142,11118,G,GC,0.863450,G,G,G,0.863450
11292,11268,C,CG,0.718024,C,C,C,0.718024
11442,11417,G,GGCA,0.205890,G,G,G,0.205890


In [6320]:
merged_df['AF_new'] = merged_df.apply(lambda x: 0 if (x['new_ref'] == x['ALT_new']) else x['AF_new'], axis=1)

merged_df.head(74)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,NaN,0,G,G,G,0.000000
1,2,C,NaN,0,C,C,C,0.000000
2,3,T,NaN,0,T,T,T,0.000000
3,4,G,NaN,0,G,G,G,0.000000
4,5,A,NaN,0,A,A,A,0.000000
...,...,...,...,...,...,...,...,...
69,70,C,NaN,0,C,C,C,0.000000
70,71,C,NaN,0,C,C,C,0.000000
71,72,T,NaN,0,T,T,T,0.000000
72,73,C,C,0.996952,A,A,C,0.003048


In [6321]:
# Ensure 'AF_old' is numeric
merged_df['AF_old'] = pd.to_numeric(merged_df['AF_old'], errors='raise')

# Filter rows where the length of 'REF_vcf' is greater than 1
copy_del_df = merged_df[merged_df['REF_vcf'].str.len() > 1].copy()

# Calculate additional columns
copy_del_df['add_len'] = copy_del_df['REF_vcf'].str.len() - 1
copy_del_df['inv_AF'] = 1 - copy_del_df['AF_old']

# Initialize a list to hold the new rows
rows = []

# Iterate through each row in the filtered DataFrame
for _, row in copy_del_df.iterrows():
    for i in range(1, row['add_len'] + 1):
        new_row = {
            'scenario': '>0.5' if row['AF_old'] > 0.5 else '<=0.5',
            'POS': row['POS'] + i,
            'add_len': row['add_len'],
            'AF': row['AF_old'],
            'inv_AF': row['inv_AF'],
            'del_POS': row['POS'] + i
        }
        rows.append(new_row)

# Create a new DataFrame from the list of new rows
new_df = pd.DataFrame(rows)

# Output the new DataFrame to check results
display(new_df.head())


,scenario,POS,add_len,AF,inv_AF,del_POS
0,>0.5,104,1,0.987559,0.012441,104
1,>0.5,538,1,0.957346,0.042654,538
2,<=0.5,634,1,0.052245,0.947755,634
3,>0.5,764,2,0.649318,0.350682,764
4,>0.5,765,2,0.649318,0.350682,765


In [6322]:
# Merge the two DataFrames on 'POS'
merged_updated_df = pd.merge(merged_df, new_df[['scenario', 'POS', 'AF', 'inv_AF']], on='POS', how='left')


merged_updated_df.head(766)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
0,1,G,NaN,0.000000,G,G,G,0.0,NaN,NaN,NaN
1,2,C,NaN,0.000000,C,C,C,0.0,NaN,NaN,NaN
2,3,T,NaN,0.000000,T,T,T,0.0,NaN,NaN,NaN
3,4,G,NaN,0.000000,G,G,G,0.0,NaN,NaN,NaN
4,5,A,NaN,0.000000,A,A,A,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
761,761,C,NaN,0.000000,C,C,C,0.0,NaN,NaN,NaN
762,762,T,NaN,0.000000,T,T,T,0.0,NaN,NaN,NaN
763,763,C,CGG,0.649318,C,C,C,0.0,NaN,NaN,NaN
764,764,G,NaN,0.000000,G,G,G,0.0,>0.5,0.649318,0.350682


In [6323]:
# merged_updated_df.loc[merged_updated_df['new_ref'] == '0', 'AF_new'].shape

In [6324]:
merged_updated_df[merged_updated_df['new_ref'] == '0']

/Users/fionachow/opt/anaconda3/envs/rl/lib/python3.9/site-packages/IPython/core/displayhook.py:281: UserWarning: Output cache limit (currently 1000 entries) hit.
Flushing oldest 200 entries.
  warn('Output cache limit (currently {sz} entries) hit.\n'


,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
103,104,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.987559,0.012441
538,538,G,NaN,0.000000,G,0,G,1.000000,>0.5,0.957346,0.042654
1674,1672,C,NaN,0.000000,C,0,C,1.000000,<=0.5,0.031593,0.968407
1675,1672,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1676,1673,T,TCCCA,0.318475,T,0,T,0.681525,>0.5,0.596132,0.403868
1677,1674,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1678,1674,C,NaN,0.000000,C,0,C,1.000000,<=0.5,0.318475,0.681525
1679,1675,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1680,1675,C,NaN,0.000000,C,0,C,1.000000,<=0.5,0.318475,0.681525
1681,1676,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868


In [6325]:
merged_updated_df[merged_updated_df['scenario'] == '>0.5']

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
103,104,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.987559,0.012441
538,538,G,NaN,0.000000,G,0,G,1.000000,>0.5,0.957346,0.042654
764,764,G,NaN,0.000000,G,G,G,0.000000,>0.5,0.649318,0.350682
765,765,G,NaN,0.000000,G,G,G,0.000000,>0.5,0.649318,0.350682
1675,1672,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1676,1673,T,TCCCA,0.318475,T,0,T,0.681525,>0.5,0.596132,0.403868
1677,1674,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1679,1675,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1681,1676,C,NaN,0.000000,C,0,C,1.000000,>0.5,0.596132,0.403868
1763,1757,T,NaN,0.000000,T,0,T,1.000000,>0.5,0.947604,0.052396


In [6326]:
merged_updated_df[merged_updated_df['scenario'] == '<=0.5'].head()

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
634,634,T,NaN,0.0,T,T,T,0.0,<=0.5,0.052245,0.947755
964,964,G,NaN,0.0,G,G,G,0.0,<=0.5,0.008979,0.991021
965,965,A,NaN,0.0,A,A,A,0.0,<=0.5,0.008979,0.991021
966,966,G,NaN,0.0,G,G,G,0.0,<=0.5,0.008979,0.991021
967,967,A,NaN,0.0,A,A,A,0.0,<=0.5,0.008979,0.991021


In [6327]:
# Update 'AF_new' based on condition
merged_updated_df.loc[merged_updated_df['new_ref'] == '0', 'AF_new'] = merged_updated_df['inv_AF']

merged_updated_df.head(104)


,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
0,1,G,NaN,0.000000,G,G,G,0.000000,NaN,NaN,NaN
1,2,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
2,3,T,NaN,0.000000,T,T,T,0.000000,NaN,NaN,NaN
3,4,G,NaN,0.000000,G,G,G,0.000000,NaN,NaN,NaN
4,5,A,NaN,0.000000,A,A,A,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
99,100,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
100,101,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
101,102,T,NaN,0.000000,T,T,T,0.000000,NaN,NaN,NaN
102,103,G,GC,0.987559,G,G,G,0.000000,NaN,NaN,NaN


In [6328]:
merged_updated_df['ALT_new'] = merged_updated_df.apply(lambda x: '0' if pd.notna(x['inv_AF']) and x['new_ref']!='0'  else x['og_ref'], axis=1)
merged_updated_df['AF_new'] = merged_updated_df.apply(lambda x: x['AF'] if pd.notna(x['inv_AF']) and x['ALT_new']=='0' else x['AF_new'], axis=1)
merged_updated_df.head(766)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new,scenario,AF,inv_AF
0,1,G,NaN,0.000000,G,G,G,0.000000,NaN,NaN,NaN
1,2,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
2,3,T,NaN,0.000000,T,T,T,0.000000,NaN,NaN,NaN
3,4,G,NaN,0.000000,G,G,G,0.000000,NaN,NaN,NaN
4,5,A,NaN,0.000000,A,A,A,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
761,761,C,NaN,0.000000,C,C,C,0.000000,NaN,NaN,NaN
762,762,T,NaN,0.000000,T,T,T,0.000000,NaN,NaN,NaN
763,763,C,CGG,0.649318,C,C,C,0.000000,NaN,NaN,NaN
764,764,G,NaN,0.000000,G,G,0,0.649318,>0.5,0.649318,0.350682


In [6329]:
merged_updated_df.drop(columns=['scenario','AF', 'inv_AF','REF_vcf', 'og_ref', 'AF_old', 'ALT_old'], inplace=True)

display(merged_updated_df.head())

,POS,new_ref,ALT_new,AF_new
0,1,G,G,0.0
1,2,C,C,0.0
2,3,T,T,0.0
3,4,G,G,0.0
4,5,A,A,0.0


In [6330]:
merge_updated_df = merged_updated_df[merged_updated_df['AF_new']!=0] #new vcf output

In [6331]:
merge_updated_df.shape

(569, 4)

In [6332]:
merge_updated_df[merge_updated_df['AF_new']>0.5].shape

(33, 4)

In [6333]:
merge_updated_df[merge_updated_df['AF_new']>0.5]

,POS,new_ref,ALT_new,AF_new
385,386,TGGCC,T,0.899331
764,764,G,0,0.649318
765,765,G,0,0.649318
1674,1672,0,C,0.968407
1678,1674,0,C,0.681525
1680,1675,0,C,0.681525
1682,1676,0,C,0.681525
1755,1749,C,C,0.570535
2493,2487,G,0,0.542124
2494,2488,C,0,0.542124
